In [1]:
import pandas as pd
from langchain_core.tools import tool
import requests
import os
from langchain_community.utilities import GoogleSerperAPIWrapper

policy_info = ''

with open("data/security_triage_policy.md", encoding='utf-8') as f:
    policy_info = f.read()

@tool
def user_info(email: str) -> str:
    "Function define to fetch the user info using email"
    print(f"Calling user_info tool (email={email})")
    df = pd.read_csv("data/users.csv")
    for _, row in df.iterrows():
        if row["user_email"] == email:
            return f"""
            Here is required user info:
            Name = {row["full_name"]}
            DEPARTMENT = {row["department"]}
            ROLE = {row["role"]}
            USUAL LOCATION = {row["usual_locations"]}
            USUAL HOURS = {row["usual_hours"]}
            USUAL DEVICE = {row["usual_device"]} 
            MFA STATUS = {row["mfa_enrolled"]}
            USUAL DATA ACCESS = {row["typical_data_access"]}
            TYPICAL RESOURCES USED = {row["typical_resources"]}
            """

@tool
def ip_info(ip: str) -> str:
    "Function define to fetch ip info using ip"
    print(f"Calling ip_info tool (ip={ip})")
    df = pd.read_csv("data/ip_reputation.csv")
    for _, row in df.iterrows():
        if row["ip"] == ip:
            return f"""
            IP Info:
            ORGANISATION = {row["org"]}
            REGION = {row['region']}
            MALICIOUS STATUS = {row['is_known_malicious']}
            PRIOR INCIDENT COUNT = {row["prior_incident_count"]}
            EXTRA NOTES (CAN BE NONE) = {row["notes"]}
            """

@tool
def notify_admin(text):
    """Send a short push notification to the user's phone."""
    print(f"Calling notify_admin tool (text={text})")
    response = requests.post(
        "https://api.pushover.net/1/messages.json",
        data={"token": os.getenv("PUSHOVER_TOKEN"), "user": os.getenv("PUSHOVER_USER"), "message": text},
    )
    response.raise_for_status()
    return "Notification Sent"

@tool
def resolve_alert(alert_id: str, status: str, resolution_notes: str) -> str:
    """Update the alert record with its final status and a summary of what was
    found and decided. status must be one of: escalated, dismissed,
    pending_human_review. Always call this exactly once, as the last step,
    for every alert you process."""
    print(f"Calling resolve_alert tool (alert_id={alert_id}, status={status})")
    df = pd.read_csv("data/alerts.csv")
    if not (df["alert_id"] == alert_id).any():
        return f"No alert found with alert_id '{alert_id}'."
    df.loc[df["alert_id"] == alert_id, "status"] = status
    df.loc[df["alert_id"] == alert_id, "resolution_notes"] = resolution_notes
    df.to_csv("data/alerts.csv", index=False)
    return f"Alert {alert_id} updated to status={status}."

@tool
def ip_internet_lookup(text: str) -> str:
    "Use this tool to lookup internet about any specific ip"
    print(f"Calling ip_internet_lookup tool (text={text})")
    search = GoogleSerperAPIWrapper()
    return search.run(text)

@tool
def check_related_alerts(ip_prefix: str = "", user_email: str = "", hours: int = 24) -> str:
    """Look up other alerts related by IP prefix or user, to check for a
    coordinated pattern. Use this before escalating or dismissing anything."""
    print(f"Calling check_related_alerts tool (ip_prefix={ip_prefix!r}, user_email={user_email!r})")
    df = pd.read_csv("data/alerts.csv")
    matches = df
    if ip_prefix:
        matches = matches[matches["ip"].str.startswith(ip_prefix)]
    if user_email:
        matches = matches[matches["user_email"] == user_email]
    if matches.empty:
        return "No related alerts found."
    return matches[["alert_id", "user_email", "ip", "timestamp", "status"]].to_string(index=False)

C:\Users\jain8\AppData\Local\Temp\ipykernel_9196\1122151148.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import GoogleSerperAPIWrapper


In [2]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph_supervisor import create_supervisor
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    ToolCallLimitMiddleware,
    ModelFallbackMiddleware,
)

import pandas as pd
import requests
import os

policy_info = ''

with open("data/security_triage_policy.md", encoding='utf-8') as f:
    policy_info = f.read()

def user_info_agent():
    return create_agent(
        model=ChatOpenAI(model='gpt-5.6-luna', reasoning_effort='none'),
        tools=[user_info],
        system_prompt="You are an amazing user analyser who get info about user and analyse the user",
        name='user-agent',
        middleware=[
            ToolCallLimitMiddleware(run_limit=5),
            ModelFallbackMiddleware("gpt-4o-mini"),
        ],
    )

def ip_info_agent():
    return create_agent(
        model=ChatOpenAI(model='gpt-5.6-luna', reasoning_effort='none'),
        tools=[ip_info],
        system_prompt="You are an amazing ip analyser who get info about ip and analyse the ip",
        name='ip-agent',
        middleware=[
            ToolCallLimitMiddleware(run_limit=5),
            ModelFallbackMiddleware("gpt-4o-mini"),
        ],
    )

def notify_user_agent():
    return create_agent(
        model=ChatOpenAI(model='gpt-5.6-luna', reasoning_effort='none'),
        tools=[notify_admin, resolve_alert],
        system_prompt=(
            "You are the notify agent. You generate an email with the final summary "
            "of the alert and send it to the admin using notify_admin. After sending "
            "the notification, you must also call resolve_alert exactly once with the "
            "alert_id, the final status (escalated, dismissed, or pending_human_review), "
            "and a resolution_notes summary of what was found and decided."
        ),
        name='notify-admin-agent',
        middleware=[
            ToolCallLimitMiddleware(run_limit=5),
            ModelFallbackMiddleware("gpt-4o-mini"),
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "resolve_alert": {
                        "allowed_decisions": ["approve", "reject"],
                        "when": lambda request: request.tool_call["args"].get("status") == "dismissed",
                    }
                }
            ),
        ],
    )

def ip_reputation_checker_agent():
    return create_agent(
        model=ChatOpenAI(model="gpt-5.6-luna", reasoning_effort='none'),
        tools=[check_related_alerts],
        name='ip-related-alert-checker',
        system_prompt='YOur task is to check the reputation of ip history to see if there is something fishy or wrong happen in past',
        middleware=[
            ToolCallLimitMiddleware(run_limit=5),
            ModelFallbackMiddleware("gpt-4o-mini"),  
        ]   
    )

def ip_internet_lookup_agent():
    return create_agent(
        model=ChatOpenAI(model="gpt-5.6-luna", reasoning_effort='none'),
        tools=[ip_internet_lookup],
        name='ip-internet-lookup',
        system_prompt='YOur task is to check the reputation of ip on the internet and find important finding related to it from internet',
        middleware=[
            ToolCallLimitMiddleware(run_limit=5),
            ModelFallbackMiddleware("gpt-4o-mini"),  
        ]   
    )  

def supervisor_agent():
    workflow = create_supervisor(
        [user_info_agent(), ip_info_agent(), notify_user_agent(), ip_reputation_checker_agent(), ip_internet_lookup_agent()],
        model=ChatOpenAI(model="gpt-5.6-luna", reasoning_effort='none'),
        prompt=(
            "You are an amazing alert researcher. You will be given info about an alert, "
            "including its alert_id. You need to analyse that alert, fetch info about the "
            "user and ip using user-agent and ip-agent respectively, also use ip reputation checker agent "
            "to see the history of same ip with the user if there is something fishy "
            "also use ip-internet-lookup agent to find the info about ip if you can't find "
            "it in the database also you can call this agent multiple time to fetch more info from internet and finally hand off "
            "to notify-admin-agent with the alert_id and your findings so it can notify the "
            "admin and record the final status and resolution notes on the alert."
            f" Here is the company policy you should use to judge the alert: {policy_info}"
        ),
    )
    return workflow.compile(checkpointer=InMemorySaver())

C:\Users\jain8\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langgraph.types import Command

def process_alerts(path="data/alerts.csv"):
    df = pd.read_csv(path)
    supervisor = supervisor_agent()
    pending = df[df["status"] == "pending"]
    for _, row in pending.iterrows():
        alert_info = f"""
        Required Info
        Alert ID = {row["alert_id"]}
        Email = {row["user_email"]}
        Time = {row["timestamp"]}
        IP = {row["ip"]}
        MFA USED = {row["mfa_used"]}
        Resource Accessed = {row["resource_accessed"]}
        Raw Description = {row["raw_description"]}
        """
        config = {"configurable": {"thread_id": row["alert_id"]}}
        result = supervisor.invoke({"messages": [{"role": "user", "content": alert_info}]}, config=config)

        if "__interrupt__" in result:
            interrupt = result["__interrupt__"][0]
            print(row["alert_id"], "- needs human approval before resolving:")
            print(interrupt.value)
            answer = input("Approve this resolution? (y/n): ").strip().lower()
            decision = {"type": "approve"} if answer == "y" else {"type": "reject", "message": "Rejected by human reviewer"}
            result = supervisor.invoke(Command(resume={"decisions": [decision]}), config=config)

        print(row["alert_id"], "->", result["messages"][-1].content)

process_alerts()

Calling user_info tool (email=jsmith@northwind.com)
Calling ip_info tool (ip=198.51.100.11)
Calling check_related_alerts tool (ip_prefix='198.51.100', user_email='jsmith@northwind.com')
Calling ip_internet_lookup tool (text=198.51.100.11)
Calling notify_admin tool (text=Alert ALT-2001 requires escalation: suspicious 03:14 login to jsmith@northwind.com via legacy IMAP without MFA. Related prior true-positive alert ALT-1001 targeted the same user from adjacent IP 198.51.100.10, indicating a repeated pattern. Note: 198.51.100.11 is RFC 5737 documentation space, so public IP reputation/geolocation is unverified; validate the original source in identity/mail/network logs. Immediately disable legacy IMAP, revoke sessions/tokens, reset password, enforce MFA, investigate mailbox rules/forwarding/OAuth and related activity, and preserve logs.)
Calling resolve_alert tool (alert_id=ALT-2001, status=escalated)
ALT-2001 -> Alert **ALT-2001** has been escalated to the administrator as **likely accou

KeyboardInterrupt: 